In [1]:
import os
import warnings
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip install transformers
!pip install sentencepiece
!pip install datasets
warnings.filterwarnings(
    "ignore", 
    message="Was asked to gather along dimension 0, but all input tensors were scalars"
)

## Local Inference on GPU 
Model page: https://huggingface.co/vesteinn/ScandiBERT

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/vesteinn/ScandiBERT)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("kaggle-scandibert")

import torch
import math
import re
from huggingface_hub import hf_hub_download
from transformers import XLMRobertaTokenizer, AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset, concatenate_datasets
from huggingface_hub import snapshot_download

model_name = "vesteinn/ScandiBERT"
labels = ["Norwegian", "Swedish", "Danish", "Non-Scandinavian"]
id2label = {i: l for i, l in enumerate(labels)}
label2id = {l: i for i, l in enumerate(labels)}

print("Downloading model components (ignoring broken tokenizer.json)...")
model_path = snapshot_download(
    repo_id="vesteinn/ScandiBERT", 
    ignore_patterns=["tokenizer.json"]
)

tokenizer = XLMRobertaTokenizer.from_pretrained(model_path)

SENTENCES_PER_CLASS = 3000
assert 1000 <= SENTENCES_PER_CLASS <= 10000, "SENTENCES_PER_CLASS is outside reasonable range!"

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels = len(labels),
    id2label = id2label,  
    label2id = label2id,
    ignore_mismatched_sizes = True,
    problem_type = "multi_label_classification"
)

def get_opus_sentences(config, lang_code, num_sentences=SENTENCES_PER_CLASS):
    print(f"Downloading {lang_code} from {config}...")
    ds = load_dataset("Helsinki-NLP/opus-100", config, split="train")
    ds = ds.filter(lambda sentence: is_dataset_clean(sentence["translation"][lang_code]))
    ds = ds.shuffle(seed=42).select(range(num_sentences))
    return [item[lang_code] for item in ds["translation"]]

def is_dataset_clean(text, instance_threshold=2, min_words=2, min_chars=6):
    bad_char_regex = r"[@#$%^&*_=+\[\]{};':\"<>/`~\\|]"
    bad_format_regex = r'[A-Z][a-z]+[A-Z]|%\s*\d|https?://'
    if not isinstance(text, str):
        return False
    if len(text.strip()) < min_chars or len(text.split()) < min_words:
        return False
    if len(re.findall(bad_char_regex, text)) >= instance_threshold:
        return False
    if re.search(bad_format_regex, text):
        return False
    return True        

def sent_based_int(percent, sent_const=SENTENCES_PER_CLASS): # Get an integer that's a % of SENTENCES_PER_CLASS
    return int(math.ceil(sent_const * percent))

sample_norwegian_all = get_opus_sentences("en-nb", "nb", sent_based_int(0.75)) + get_opus_sentences("en-nn", "nn", sent_based_int(0.25))
sample_swedish_all = get_opus_sentences("en-sv", "sv")
sample_danish_all = get_opus_sentences("da-en", "da")
nonscandi_configs = [("en-es", "en"), ("en-es", "es"), ("en-is", "is"), ("de-en", "de"), ("en-nl", "nl"), ("af-en", "af")]
sample_nonscandi_all = []
for config, code in nonscandi_configs:
    sample_nonscandi_all.extend(get_opus_sentences(config, code, sent_based_int(1 / len(nonscandi_configs))))
print(f"Norwegian corpora collected: {len(sample_norwegian_all)} sentences.")
print(f"Swedish corpora collected: {len(sample_swedish_all)} sentences.")
print(f"Danish corpora collected: {len(sample_danish_all)} sentences.")
print(f"Non-Scandinavian corpora collected: {len(sample_nonscandi_all)} sentences, with {len(sample_nonscandi_all) / len(nonscandi_configs)} sentences per language.")

test_text = "Dette er en norsk setning."
inputs = tokenizer(test_text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
print("\nZero-Shot Classification Probabilities: ")
probs = torch.sigmoid(outputs.logits)
for label, prob in zip(labels, probs[0]): # True "zero-shot", random classification test
    print(f"{label}: {prob.item():.4f}")

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: vesteinn/ScandiBERT
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


README.md: 0.00B [00:00, ?B/s]

en-nb/test-00000-of-00001.parquet:   0%|          | 0.00/223k [00:00<?, ?B/s]

en-nb/train-00000-of-00001.parquet:   0%|          | 0.00/10.2M [00:00<?, ?B/s]

... (59 outputs left out)

Generating train split:   0%|          | 0/275512 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/275512 [00:00<?, ? examples/s]

Norwegian corpora collected: 3000 sentences.
Swedish corpora collected: 3000 sentences.
Danish corpora collected: 3000 sentences.
Non-Scandinavian corpora collected: 3000 sentences, with 500.0 sentences per language.

Zero-Shot Classification Probabilities: 
Norwegian: 0.5139
Swedish: 0.4193
Danish: 0.5362
Non-Scandinavian: 0.4820


In [3]:
from datasets import Dataset, concatenate_datasets, Sequence, Value, Features
from sklearn.model_selection import train_test_split

def smoothed_values(target_labels, labels_list=labels, SMOOTH_VAL=0.1):
    if isinstance(target_labels, str):
        target_labels = [target_labels] # Accepts either a lang string or a lang list
    assert all(t in labels_list for t in target_labels), "Invalid target label(s)."
    
    num_labels = len(labels_list)
    num_targets = len(target_labels)
    new_list = [None] * num_labels

    for i, curr_label in enumerate(labels_list):
        is_target = curr_label in target_labels
        is_scandi = curr_label != "Non-Scandinavian"

        if is_target:
            new_list[i] = 1 - SMOOTH_VAL 
        elif is_scandi:
            no_da_pair = {"Norwegian", "Danish"}
            if no_da_pair.issubset(set(target_labels)):
                new_list[i] = SMOOTH_VAL / 3  # Swedish off-label
            else:
                da_no_cross = ("Norwegian" in target_labels and curr_label == "Danish") or ("Danish" in target_labels and curr_label == "Norwegian")
                new_list[i] = SMOOTH_VAL / (1.5 if da_no_cross else 3)
        else:
            new_list[i] = SMOOTH_VAL / 4  # Non-Scandinavian always low

    return new_list
    

EVAL_DS_SPLIT = 0.15

norwegian_train, norwegian_eval = train_test_split(
    sample_norwegian_all, test_size=EVAL_DS_SPLIT, random_state=42
)
swedish_train, swedish_eval = train_test_split(
    sample_swedish_all, test_size=EVAL_DS_SPLIT, random_state=42
)
danish_train, danish_eval = train_test_split(
    sample_danish_all, test_size=EVAL_DS_SPLIT, random_state=42
)
nonscandi_train, nonscandi_eval = train_test_split(
    sample_nonscandi_all, test_size=EVAL_DS_SPLIT, random_state=42
)

# Very tiny makeshift corpus of sentences valid in multiple languages to avoid overconfidence in one lang.
all_scandi_sent = ["En dag i livet", "Vi har en bil.", "Han har en god plan.", "Herregud!", "Studenten ser en musiker.", "En normal person i en varm taxi kan se en film."]
da_no_cross_sent = ["Det kan jeg ikke.", "Dette er en fugl.", "Han spiste brødet.", "Hvem er du?", "Tyskland er et land.", "Middagen er klar!"]
all_scandi_labels = [smoothed_values(["Norwegian", "Swedish", "Danish"])] * len(all_scandi_sent)
da_no_cross_labels = [smoothed_values(["Norwegian", "Danish"])] * len(da_no_cross_sent)

CROSS_SCANDI_MULTIPLIER = max(1, int(sent_based_int(0.75) * 0.05 / (len(all_scandi_sent) + len(da_no_cross_sent))))

label_features = Features({"text": Value("string"), "labels": Sequence(Value("float32"))})

all_scandi_train, all_scandi_eval = train_test_split(
    list(zip(all_scandi_sent, all_scandi_labels)),
    test_size=EVAL_DS_SPLIT, random_state=42
)
da_no_cross_train, da_no_cross_eval = train_test_split(
    list(zip(da_no_cross_sent, da_no_cross_labels)),
    test_size=EVAL_DS_SPLIT, random_state=42
)
ms_train_texts, ms_train_labels = zip(*all_scandi_train)
ms_eval_texts, ms_eval_labels = zip(*all_scandi_eval)
da_no_cross_train_texts, da_no_cross_train_labels = zip(*da_no_cross_train)
da_no_cross_eval_texts, da_no_cross_eval_labels = zip(*da_no_cross_eval)

dataset_norwegian_train = Dataset.from_dict({"text": norwegian_train, "labels": [smoothed_values("Norwegian")] * len(norwegian_train)}, features=label_features)
dataset_swedish_train = Dataset.from_dict({"text": swedish_train, "labels": [smoothed_values("Swedish")] * len(swedish_train)}, features=label_features)
dataset_danish_train = Dataset.from_dict({"text": danish_train, "labels": [smoothed_values("Danish")] * len(danish_train)}, features=label_features)
dataset_nonscandi_train = Dataset.from_dict({"text": nonscandi_train, "labels": [smoothed_values("Non-Scandinavian")] * len(nonscandi_train)}, features=label_features)
dataset_multiscandi_train = Dataset.from_dict({"text": list(ms_train_texts) * CROSS_SCANDI_MULTIPLIER, "labels": list(ms_train_labels) * CROSS_SCANDI_MULTIPLIER}, features=label_features)
dataset_da_no_cross_train = Dataset.from_dict({"text": list(da_no_cross_train_texts) * CROSS_SCANDI_MULTIPLIER, "labels": list(da_no_cross_train_labels) * CROSS_SCANDI_MULTIPLIER}, features=label_features)
train_dataset = concatenate_datasets([dataset_norwegian_train, dataset_swedish_train, dataset_danish_train, dataset_nonscandi_train, dataset_multiscandi_train, dataset_da_no_cross_train]).shuffle(seed=42)

dataset_norwegian_eval = Dataset.from_dict({"text": norwegian_eval, "labels": [smoothed_values("Norwegian")] * len(norwegian_eval)}, features=label_features)
dataset_swedish_eval = Dataset.from_dict({"text": swedish_eval, "labels": [smoothed_values("Swedish")] * len(swedish_eval)}, features=label_features)
dataset_danish_eval = Dataset.from_dict({"text": danish_eval, "labels": [smoothed_values("Danish")] * len(danish_eval)}, features=label_features)
dataset_nonscandi_eval = Dataset.from_dict({"text": nonscandi_eval, "labels": [smoothed_values("Non-Scandinavian")] * len(nonscandi_eval)}, features=label_features)
dataset_multiscandi_eval = Dataset.from_dict({"text": list(ms_eval_texts) * CROSS_SCANDI_MULTIPLIER, "labels": list(ms_eval_labels) * CROSS_SCANDI_MULTIPLIER}, features=label_features)
dataset_da_no_cross_eval = Dataset.from_dict({"text": list(da_no_cross_eval_texts) * CROSS_SCANDI_MULTIPLIER, "labels": list(da_no_cross_eval_labels) * CROSS_SCANDI_MULTIPLIER}, features=label_features)
eval_dataset = concatenate_datasets([dataset_norwegian_eval, dataset_swedish_eval, dataset_danish_eval, dataset_nonscandi_eval, dataset_multiscandi_eval, dataset_da_no_cross_eval]).shuffle(seed=42)

print(train_dataset[:20]) # Test to see if dataset is selected + labeled properly

def tokenize_dataset(select_dataset):
    return tokenizer(select_dataset["text"], truncation=True, max_length=512)

print(f"{len(train_dataset)} Training samples | {len(eval_dataset)} Eval samples")

train_dataset = train_dataset.map(tokenize_dataset, batched=True)
eval_dataset = eval_dataset.map(tokenize_dataset, batched=True)
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

{'text': ['Jag tänkte inte ens på dig.', 'Generaldirektor für Landwirtschaft und ländliche Entwicklung', 'Helling: Storleiken på penselen er avhengig av hellinga på penselen.', 'Gir tilgang til bursdager og bryllupsdager for kontakter i adresseboka, som kalenderhendelserName', '- Leroy, han hjälper oss.', 'Je kunt ze niet door Marokko slepen in het gezelschap van ontsnapte criminelen.', 'Qatar- rialer', 'Vi har stemt ja til denne betænkning, eftersom den har et godt indhold.', 'Ik kan hem niet eerder horen dan morgen als zijn advocaat er is.', '- Nej, Michael!', 'Jeg er kun et menneske–', 'Hoorde dat je vrij was.', 'Vad tror du att de kommer göra?', 'Verander agtergrond kleur vir huidiglik gekose teks .', 'Hvit bakgrunn, 640 x 480Comment', '- Alle okay?', 'Det er lite rom for tvil.', 'Græd ikke.', 'El dictamen que pide el artículo 109 J del Tratado, dictamen emitido sobre un texto del Consejo Ecofin, en ese caso, transmitido al Consejo reunido en el nivel de los Jefes de Estado y de Go

Map:   0%|          | 0/10290 [00:00<?, ? examples/s]

Map:   0%|          | 0/1818 [00:00<?, ? examples/s]

In [4]:
import numpy as np
from sklearn.metrics import f1_score
def compute_metrics(eval_training, labels_list=labels): # Report each class' performance for trainng
    logits, soft_labels = eval_training
    probs = 1 / (1 + np.exp(-logits)) # Manual sigmoid
    preds = (probs > 0.5).astype(int)
    hard_labels = (soft_labels > 0.5).astype(int)

    f1_per_class = f1_score(hard_labels, preds, average=None, zero_division=0)
    macro_f1 = f1_score(hard_labels, preds, average="macro", zero_division=0)

    result = {"macro_f1": macro_f1}
    for name, f1 in zip(labels_list, f1_per_class):
        result[f"f1_{name.lower().replace('-', '_')}"] = f1
    return result

print(tokenizer.tokenize("Dette er tomt."))

['▁D', 'et', 'te', '▁er', '▁to', 'mt', '.']


In [5]:
import re
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Phase 1: Freezing ScandiBERT \"tail\" for training...")
for param in model.roberta.parameters():
    param.requires_grad = False

training_args_phase1 = TrainingArguments(
    output_dir="./scandiprob_phase1",
    learning_rate=1e-3,                # Fast!      
    per_device_train_batch_size=8,
    num_train_epochs=2,                     
    weight_decay=0.01,
    fp16=False,
    report_to="none",
    dataloader_pin_memory=False,
    logging_steps=sent_based_int(0.1),
    eval_strategy="epoch"
)

trainer_phase1 = Trainer(
    model=model,
    args=training_args_phase1,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Starting Phase 1 Training...")
trainer_phase1.train()


print("\nPhase 2: Unfreezing for full fine-tuning...")
for param in model.roberta.parameters():
    param.requires_grad = True

training_args_phase2 = TrainingArguments(
    output_dir="./scandiprob_phase2",
    learning_rate=2e-5,                      # = 0.00002, "standard"
    per_device_train_batch_size=8,
    num_train_epochs=4,                      
    weight_decay=0.01,
    warmup_steps=sent_based_int(0.15),                        
    fp16=False,
    report_to="none",
    dataloader_pin_memory=False,
    logging_steps=sent_based_int(0.1),
    eval_strategy="epoch",
    save_strategy="epoch"               
)

trainer_phase2 = Trainer(
    model=model,
    args=training_args_phase2,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Starting Phase 2 Training...")
trainer_phase2.train()
trainer_phase2.save_model("./scandiprob_full")
print("Training complete and model saved!")

def nonscandi_penalty(text): # Regex scoring for negative class
    if not text:
        return 0.0

    scandi_keyboard = r"[a-zA-ZæøåÆØÅäöÄÖéÉ0-9 !@#$%^&*()\-_=+\[\]{};':\",.<>?/`~\\|]"
    scandi_matches = re.findall(scandi_keyboard, text)
    nonscandi_percent = (1 - (len(scandi_matches) / len(text)))
    return nonscandi_percent

def da_no_cross_penalty(text): # WiP Linguistic heuristic scoring for differences between Norwegian and Danish
    pass


def ScandiProb(text): # Actually showing the fine-tuned model some text to classify
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    model.eval()
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    print("\n")
    raw_probs = torch.sigmoid(outputs.logits)[0]

    nonscandi_ratio = nonscandi_penalty(text)
    
    
    final_probs = []
    skew_effect = False
    
    for i, label in enumerate(labels):
        prob = raw_probs[i].item()
        
        if label in ["Norwegian", "Swedish", "Danish"]:
            adjusted = prob * (1.0 - nonscandi_ratio)
        else: # Scale up/down by % of non-Scandi chars
            adjusted = prob + ((1.0 - prob) * nonscandi_ratio)
            
        final_probs.append(adjusted)
    
        if (round(adjusted, 4) != round(prob, 4)):
            skew_effect = True

    if skew_effect:
        print("Pre-Adjustment Probabilities: ")
        for label, prob in zip(labels, raw_probs): # Print pre-regex-skew results!
            print(f"{label}: {prob:.4f}")
    print("\nPost-Adjustment Probabilities: " if skew_effect else "Post-Training Probabilities")
    for label, prob in zip(labels, final_probs): # Print post-regex-skew results!
        print(f"{label}: {prob:.4f}")

Phase 1: Freezing ScandiBERT "tail" for training...
Starting Phase 1 Training...


Epoch,Training Loss,Validation Loss,Macro F1,F1 Norwegian,F1 Swedish,F1 Danish,F1 Non Scandinavian
1,0.534742,0.508273,0.151772,0.000000,0.000000,0.000000,0.607088
2,0.497340,0.472502,0.162199,0.000000,0.042644,0.000000,0.606154


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (6 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete and model saved!


In [6]:
def scandiprob_test(input_text):
    print(input_text)
    ScandiProb(input_text)
    print("\n")

scandiprob_test("Dette er en norsk setning.")
scandiprob_test("Detta är en svensk mening.")
scandiprob_test("Dette er en dansk sætning.")

scandiprob_test("Þetta er íslensk setning.") # Icelandic, in pretraining but not in training
scandiprob_test("Dies ist ein deutscher Satz.") # German, in negative class training but not pretraining
scandiprob_test("Šis ir teikums latviešu valodā.") # Latvian, not in pretraining or training, but should be partially flagged by regex
scandiprob_test("これは日本語の文章です。") # Japanese, not in pretraining or training, but should be fully flagged by regex

scandiprob_test("En dag i livet") # Should be valid in Norwegian, Swedish, and Danish; from University of Oslo 2025 SLIDE paper
scandiprob_test("Der stod ingen steder i Mai Buchs eksamenspapirer, at hun skulle havne pa et teater.") # Norwegian vs. Danish test, from SLIDE paper; Danish *should* prevail

# Excerpt from an Aftenposten (Norwegian) news article
scandiprob_test("Onsdag diskuterer EU hastetiltak som kan få ned forbruket av importert olje og gass. Ett av forslagene bringer frem minner fra pandemi-tiden: å oppfordre bedrifter til å innføre obligatorisk hjemmekontor en dag i uken, viser et lekket utkast sett av Financial Times og Politico.")
# Excerpt from a Dagens Nyheter (Swedish) news article
scandiprob_test("Det var en ansatt rikspolischef som klev ut genom den praktfulla dörren till justitieutskottets sammanträdessal i riksdagshuset. Petra Lundh var kallad för att svara på frågor om Polismyndighetens användning av AI-verktyg och eventuella affärer med den kontroversiella, amerikanska techjätten Palantir.")
# Excerpt from a Politiken (Danish) news article
scandiprob_test("Et loft over huslejen vil gøre det værre for lejerne. Det er den noget aparte påstand i et indlæg fra Christian Curdt-Christiansen (Pol 2.4.). Skribenten, der præsenterer sig selv som en 25-årig studerende, burde nok lige have deklareret, at han ifølge egne oplysninger er på lønningslisten hos lobbyorganisationen Bygherreforeningen [...]")

Dette er en norsk setning.


Post-Training Probabilities
Norwegian: 0.4321
Swedish: 0.0126
Danish: 0.8021
Non-Scandinavian: 0.0126


Detta är en svensk mening.


Post-Training Probabilities
Norwegian: 0.0298
Swedish: 0.9255
Danish: 0.0283
Non-Scandinavian: 0.0194


Dette er en dansk sætning.


Post-Training Probabilities
Norwegian: 0.0777
Swedish: 0.0242
Danish: 0.9242
Non-Scandinavian: 0.0169


Þetta er íslensk setning.


Pre-Adjustment Probabilities: 
Norwegian: 0.0324
Swedish: 0.0267
Danish: 0.0347
Non-Scandinavian: 0.9242

Post-Adjustment Probabilities: 
Norwegian: 0.0298
Swedish: 0.0245
Danish: 0.0319
Non-Scandinavian: 0.9302


Dies ist ein deutscher Satz.


Post-Training Probabilities
Norwegian: 0.0280
Swedish: 0.0317
Danish: 0.0281
Non-Scandinavian: 0.9281


Šis ir teikums latviešu valodā.


Pre-Adjustment Probabilities: 
Norwegian: 0.0289
Swedish: 0.0328
Danish: 0.0278
Non-Scandinavian: 0.9224

Post-Adjustment Probabilities: 
Norwegian: 0.0261
Swedish: 0.0296
Danish: 0.0251
Non